In [ ]:
# # 九曜：天墟 LF01 — 國漫配音 第 4 版（Qwen3-TTS，Kaggle）2026-09-30
#
# 咖哩試聽第 3 版：机遙太像小孩、郁岑燁不夠高冷、厲若楓與院方執事變粵語；江祈璟、赤瞳妖將定案；齊衡烈用 EP01 v1 的聲音。
# 本版：
#  1. 机遙（改成熟青年音）、郁岑燁（改高冷）、厲若楓、院方執事：VoiceDesign 直接唸，提示詞與台詞一律轉**簡體**
#     （假設：繁體字讓模型往港式／粵語偏）；厲若楓、院方執事另出**繁體**對照組，同種子比較。
#  2. 齊衡烈：EP01 Qwen v1 參考聲音（咖哩認可）用 Base 照唸 LF01 新增 3 句，每句 3 次。
# 由 Claude 經 Kaggle API 推送執行；MP3 以 base64 印進執行紀錄供 Claude 取回。


In [ ]:
import subprocess
print(subprocess.run("pip install -q -U qwen-tts soundfile 2>&1 | tail -3; nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)


In [ ]:
import json, os, base64
ROWS = json.loads(r'''[["LF01-A05_机遙_簡", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。男主角，二十岁出头的青年男声，声音成熟稳重、中低音、有厚度，绝对不是小孩或少年音。沉静内敛、话少，开口前略停一下；语气平静，带一点审慎的疑问，骨子里冷静有主见。", "应该是。"], ["LF01-A08_机遙_簡", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。男主角，二十岁出头的青年男声，声音成熟稳重、中低音、有厚度，绝对不是小孩或少年音。沉静内敛、话少，开口前略停一下；语气平静，带一点审慎的疑问，骨子里冷静有主见。", "你们都看得出来？"], ["LF01-A06_郁岑燁_簡", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。二十岁天才剑修，高冷孤傲的青年男声，音色清冷干净、像冰一样，音调中等偏高。语速偏慢、语气淡漠疏离，带著居高临下的轻蔑，懒得多说一个字，句尾轻轻下压，一句话就让人觉得被看不起。", "你看都没看他出手，怎么知道他能打？"], ["LF01-A07_郁岑燁_簡", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。二十岁天才剑修，高冷孤傲的青年男声，音色清冷干净、像冰一样，音调中等偏高。语速偏慢、语气淡漠疏离，带著居高临下的轻蔑，懒得多说一个字，句尾轻轻下压，一句话就让人觉得被看不起。", "……无聊。"], ["LF01-A08_厲若楓_簡", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。二十岁弓手青年，低沉柔和、带一点气声的男声，音量小、语速慢，句与句之间有明显停顿。冷静、专注、观察入微，说话像在陈述事实。", "别急。他的脚步有点乱。"], ["LF01-A08_厲若楓_繁", "熱血玄幻國漫（3D修仙動畫）的配音，說標準普通話，專業配音演員的表演腔，咬字清楚。二十歲弓手青年，低沉柔和、帶一點氣聲的男聲，音量小、語速慢，句與句之間有明顯停頓。冷靜、專注、觀察入微，說話像在陳述事實。", "別急。他的腳步有點亂。"], ["LF01-A03_院方執事_簡", "热血玄幻国漫（3D修仙动画）的配音，说标准普通话，专业配音演员的表演腔，咬字清楚。学院登记处的中年执事，平稳清楚的男声，音调中等，公事公办、语气平淡有礼，像在唱名点名，一字一句念得清楚。", "青岚新契修，机遥。东廊，第七室。"], ["LF01-A03_院方執事_繁", "熱血玄幻國漫（3D修仙動畫）的配音，說標準普通話，專業配音演員的表演腔，咬字清楚。學院登記處的中年執事，平穩清楚的男聲，音調中等，公事公辦、語氣平淡有禮，像在唱名點名，一字一句念得清楚。", "青嵐新契修，机遙。東廊，第七室。"]]''')
QI = json.loads(r'''{"ref_text": "今天天氣不錯，我們慢慢走過去吧，路上還可以再聊一聊。", "b64": "..."}''')
QI_LINES = [["LF01-A11_齊衡烈", "習慣就好。"], ["LF01-A12_齊衡烈", "喂，新來的。我叫齊衡烈。你呢？"], ["LF01-B01_齊衡烈", "它們在逃？"]]
SEEDS = [11, 22, 33, 44]; QI_TAKES = 3
OUT = "/kaggle/working/voice_guoman"; os.makedirs(OUT, exist_ok=True)
open("/tmp/qi.flac", "wb").write(base64.b64decode(QI["b64"]))
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/tmp/qi.flac", "-ar", "24000", "/tmp/qi.wav"], check=True)


In [ ]:
# VoiceDesign 直接唸（ROWS：名稱、提示詞、台詞）
import torch, soundfile as sf, gc
from qwen_tts import Qwen3TTSModel
design = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=torch.float32, attn_implementation="sdpa")
for name, prompt, text in ROWS:
    for s in SEEDS:
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
        wavs, sr = design.generate_voice_design(text=text, language="Chinese", instruct=prompt)
        sf.write(f"{OUT}/{name}_s{s}.wav", wavs[0][: int(sr * 15)], sr)
        print(name, f"s{s}", f"{len(wavs[0])/sr:.2f}s", text)
del design; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# 齊衡烈：EP01 v1 參考聲音照唸（Base）
clone = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0", dtype=torch.float32, attn_implementation="sdpa")
w, sr0 = sf.read("/tmp/qi.wav")
prompt = clone.create_voice_clone_prompt(ref_audio=(w, sr0), ref_text=QI["ref_text"])
for name, text in QI_LINES:
    for k in range(1, QI_TAKES + 1):
        try:
            wavs, sr = clone.generate_voice_clone(text=text, language="Chinese", voice_clone_prompt=prompt, max_new_tokens=12 * 10)
        except TypeError:
            wavs, sr = clone.generate_voice_clone(text=text, language="Chinese", voice_clone_prompt=prompt)
        sf.write(f"{OUT}/{name}_t{k}.wav", wavs[0][: int(sr * 10)], sr)
        print(name, f"t{k}", f"{len(wavs[0])/sr:.2f}s", text)


In [ ]:
# 乾聲＋混音版 MP3，並以 base64 印進執行紀錄（Claude 從雲端抓不到 Output 檔）
import glob, base64, shutil
MIX = "equalizer=f=110:t=q:w=1:g=5,equalizer=f=3500:t=q:w=1.5:g=2,acompressor=threshold=-20dB:ratio=4:attack=5:release=120:makeup=4,aecho=0.8:0.6:60|120:0.25|0.15,loudnorm=I=-16"
dst = "/kaggle/working/voice_guoman_tiny"; os.makedirs(dst, exist_ok=True)
for w in sorted(glob.glob(f"{OUT}/*.wav")):
    b = f"{dst}/{os.path.basename(w)[:-4]}"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", w, "-ac", "1", "-b:a", "64k", b + ".mp3"], check=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", w, "-af", MIX, "-ac", "1", "-b:a", "64k", b + "_混音.mp3"], check=True)
shutil.make_archive("/kaggle/working/LF01_voice_guoman_tiny", "zip", dst)
for m in sorted(glob.glob(f"{dst}/*.mp3")):
    print(f"@@MP3 {os.path.basename(m)} {base64.b64encode(open(m, 'rb').read()).decode()}")
